# Bear_BBR Signal - Data Prep

## Import Libs

In [ ]:
import pandas as pd 
import numpy as np
import os
from pandas import DataFrame, Series
import plotly.graph_objects as go

## Read CSV price data and get DataFrame

In [648]:
PATH = os.getcwd()
FILE = "../output/FE_V2_GBPUSD_15mins_1yr_End_20260311.csv"
df = pd.read_csv(FILE)
df["Date"] = pd.DatetimeIndex(df["Date"], tz="US/Eastern")
df.set_index("Date", inplace=True)
df.info()

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 24580 entries, 2025-03-11 17:15:00-04:00 to 2026-03-11 16:45:00-04:00
Data columns (total 47 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Open           24580 non-null  float64
 1   High           24580 non-null  float64
 2   Low            24580 non-null  float64
 3   Close          24580 non-null  float64
 4   Idx            24580 non-null  int64  
 5   Body           24580 non-null  float64
 6   Range          24580 non-null  float64
 7   UWick          24580 non-null  float64
 8   LWick          24580 non-null  float64
 9   Close_%High    24580 non-null  float64
 10  Open_%High     24580 non-null  float64
 11  Iday_Idx       24580 non-null  int64  
 12  Iday_High      24580 non-null  float64
 13  Iday_Low       24580 non-null  float64
 14  Iday_Range     24580 non-null  float64
 15  Close_%DHigh   24580 non-null  float64
 16  Open_%DHigh    24580 non-null  float64
 17  Yda

## Add extra features

In [649]:
def get_yra_std(row: Series, df: DataFrame, n: int):
    std = df["YRA_Diff"].at_time("17:15").iloc[row["Day_Idx"]-n+1:row["Day_Idx"]+1].std()
    return std

pd.options.display.max_rows = 100
pd.options.display.max_columns = 100
df["Yday_Range"] = df.eval("Yday_High - Yday_Low")
df["YRA_Diff"] = df.apply(lambda x: x["Yday_Range"] - x["ADR"], axis=1)
df["YRA_STD"] = df.apply(get_yra_std, axis=1, args=[df.copy(), 30])


In [650]:
bbu_reversal = df
bbu_reversal.info()

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 24580 entries, 2025-03-11 17:15:00-04:00 to 2026-03-11 16:45:00-04:00
Data columns (total 50 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Open           24580 non-null  float64
 1   High           24580 non-null  float64
 2   Low            24580 non-null  float64
 3   Close          24580 non-null  float64
 4   Idx            24580 non-null  int64  
 5   Body           24580 non-null  float64
 6   Range          24580 non-null  float64
 7   UWick          24580 non-null  float64
 8   LWick          24580 non-null  float64
 9   Close_%High    24580 non-null  float64
 10  Open_%High     24580 non-null  float64
 11  Iday_Idx       24580 non-null  int64  
 12  Iday_High      24580 non-null  float64
 13  Iday_Low       24580 non-null  float64
 14  Iday_Range     24580 non-null  float64
 15  Close_%DHigh   24580 non-null  float64
 16  Open_%DHigh    24580 non-null  float64
 17  Yda

# Bear_BBR Signal Analysis

## Bearish BBR Metrics Function

Return a new dataframe with all occrrences of Bear_BBR signal and get additional metrics such as:
- Entry Price
- Max Gain / Loss
- Gain/Loss to:
    - SMA16,
    - SMA32,
    - BB_Lower,
    - SigLow,
    - First_Bullish_Signal (Bull_0)
    - Bull_Engulf
    - Intraday_Low_Reversal (ILR)
- Whether S/R was tested or broken
- Whether other bearish pattern occurred at the same time

In [628]:

def bearish_bbr_metrics(df: DataFrame, bear_bbr_signal_idx: list):
    """Return a new dataframe with signal metrics"""

    signal = []
    for signal_idx in bear_bbr_signal_idx:
        row = df.iloc[signal_idx]
        s = {}
        START = row.name
        TD = pd.Timedelta(minutes=15)
        s["Date"] = START
        s["Entry_Price"] = row["Close"] # signal price
        
        day = row["Day_Idx"]
        window = df.query(f"Day_Idx == {day}") # only get df data from the same FX day as signal
        before_signal_window = window[:START-TD] # get df data from start of day until just before signal
        signal_window = window[START:] # get df from signal until end of day (EOD)
        
        Max_High = signal_window[START+TD:]["High"].max() # Max high from signal until EOD
        Min_Low = signal_window[START+TD:]["Low"].min() # Min low from signal until EOD
        s["D_Max_Up"] = s["Entry_Price"] - Max_High # Max price gain from signal until EOD
        s["D_Max_Down"] = s["Entry_Price"] - Min_Low # Max price loss from signal until EOD
        D_Close = signal_window.iloc[-1]["Close"] # Closing price at EOD
        s["D_Gain"] = s["Entry_Price"] - D_Close # Gain at EOD from signal 


        if signal_window.query(f"Close > {row["High"]}").empty is False:
            # If price closed above signal price within the day, 
            # get the dataframe between start and failure time
            EXIT = signal_window[START:].query(f"Close > {row["High"]}").iloc[0].name
            entry_window = signal_window[START:EXIT]
        else:
            # If price didn't close above signal price within the day
            # get the dataframe from signal time until EOD
            entry_window = signal_window

        # Did the price go higher than the signal candle high
        if entry_window[START+TD:].query(f"High >= {row["High"]}").empty is False:
            s["Stop_Pips"] = s["Entry_Price"] - row["High"]
        else:
            s["Stop_Pips"] = 0
        
        s["Max_Up"] = s["Entry_Price"] - entry_window[START+TD:]["Close"].max()
        s["Max_Down"] = s["Entry_Price"] - entry_window[START+TD:]["Low"].min()
        

        # If the price fell to the SMA16 after the signal
        # calcuate the gain
        below_sma16 = entry_window.query("Low < SMA16")
        if below_sma16.empty is False:
            s["To_SMA16"] = s["Entry_Price"] - below_sma16.iloc[0]["SMA16"] 
        else: 
            s["To_SMA16"] = 0

        # If the price fell to the SMA32 after the signal
        # calcuate the gain
        below_sma32 = entry_window.query("Low < SMA32")
        if below_sma32.empty is False:
            s["To_SMA32"] = s["Entry_Price"] - below_sma32.iloc[0]["SMA32"]
        else:
            s["To_SMA32"] = 0

        # If the price fell to the BB_Lower_16_2 after the signal
        # calcuate the gain
        below_bbl = entry_window.query("Low < BB_Lower_16_2")
        if below_bbl.empty is False:
            s["To_BBL_16_2"] = s["Entry_Price"] - below_bbl.iloc[0]["BB_Lower_16_2"]
        else:
            s["To_BBL_16_2"] = 0

        # If the price fell to the Sig_Low after the signal
        # calcuate the gain
        below_sig_low = entry_window.query("Low < Sig_Low")
        if below_sig_low.empty is False:
            s["To_Sig_Low"] = s["Entry_Price"] - below_sig_low.iloc[0]["Sig_Low"]
        else:
            s["To_Sig_Low"] = 0

        below_sr = entry_window.query("S_R > 3")["Sig_Low"].unique()
        for i in range(len(below_sr)):
            s[f"To_S_{i}"] = s["Entry_Price"] - below_sr[i]


        # Check if bullish signals occured during winddow
        NULL_Timestamp = pd.Timestamp(0, tz="US/Eastern")
        to_bull_bbr = entry_window.query("Bull_BBR == True")
        if to_bull_bbr.empty is False:
            s["To_Bull_BBR"] = s["Entry_Price"] - to_bull_bbr.iloc[0]["Close"]
            to_bull_bbr_time_stamp = to_bull_bbr.iloc[0].name
        else:
            s["To_Bull_BBR"] = 0
            to_bull_bbr_time_stamp = NULL_Timestamp

        to_bull_engulf = entry_window.query("Bull_Engulf == True")
        if to_bull_engulf.empty is False:
            s["To_Bull_Engulf"] = s["Entry_Price"] - to_bull_engulf.iloc[0]["Close"]
            to_bull_engulf_time_stamp = to_bull_engulf.iloc[0].name
        else:
            s["To_Bull_Engulf"] = 0
            to_bull_engulf_time_stamp = NULL_Timestamp

        to_ilr = entry_window.query("ILR == True")
        if to_ilr.empty is False:
            s["To_ILR"] = s["Entry_Price"] - to_ilr.iloc[0]["Close"]
            to_ilr_time_stamp = to_ilr.iloc[0].name
        else:
            s["To_ILR"] = 0
            to_ilr_time_stamp = NULL_Timestamp

        first_bullish_signal = min([to_bull_bbr_time_stamp, to_bull_engulf_time_stamp, to_ilr_time_stamp])
        if first_bullish_signal > NULL_Timestamp:
            s["To_Bull_0"] = s["Entry_Price"] - entry_window.loc[first_bullish_signal]["Close"]
        else:
            s["To_Bull_0"] = 0

        # how man times was resistance tested before signal
        r_test = before_signal_window.query("S_R == 1")["S_R"].count() 
        # how man times was resistance broken before signal
        r_break = before_signal_window.query("S_R == 2")["S_R"].count()
        # how man times was support tested before signal
        s_test = before_signal_window.query("S_R == 3")["S_R"].count()
        # how man times was support broken before signal
        s_break = before_signal_window.query("S_R == 4")["S_R"].count()

        h1_range = window[:START].iloc[-4:]["High"].max() - window[:START].iloc[-4:]["Low"].min()


        # append other useful df metrics to new df
        s["Range"] = row["Range"]
        s["ATR"] = row["ATR"]
        s["Close_%High"] = row["Close_%High"]
        s["SMA_Trend"] = row["SMA_Trend"]
        s["SMA16_Slope"] = row["SMA16_Slope"]
        s["SMA32_Slope"] = row["SMA32_Slope"]
        s["RSI"] = row["RSI"]
        s["IHR"] = row["IHR"]
        s["Dark_Cloud"] = row["Dark_Cloud"]
        s["Shooting_Star"] = row["Shooting_Star"]
        s["Bear_Engulf"] = row["Bear_Engulf"]
        s["Iday_Range"] = row["Iday_Range"]
        s["ADR"] = row["ADR"]
        s["Close_%DHigh"] = row["Close_%DHigh"]
        s["SMA16"] = row["SMA16"]
        s["SMA32"] = row["SMA32"]
        s["S_R"] = row["S_R"]
        s["YRA_Diff"] = row["YRA_Diff"]
        s["YRA_STD"] = row["YRA_STD"]
        s["R_Tested"] = r_test
        s["R_Broken"] = r_break
        s["S_Tested"] = s_test
        s["S_Broken"] = s_break
        s["H1_Range"] = h1_range
        

        signal.append(s)
    sdf = pd.DataFrame(signal)
    return sdf


### Get Bear_BBR Metrics

In [651]:
signal_idx_list = bbu_reversal.query("Bear_BBR == True")["Idx"].to_list()
bearish_bbr_df = bearish_bbr_metrics(bbu_reversal, signal_idx_list)
bearish_bbr_df.tail(10)

,Date,Entry_Price,D_Max_Up,D_Max_Down,D_Gain,Stop_Pips,Max_Up,Max_Down,To_SMA16,To_SMA32,To_BBL_16_2,To_Sig_Low,To_Bull_BBR,To_Bull_Engulf,To_ILR,To_Bull_0,Range,ATR,Close_%High,SMA_Trend,SMA16_Slope,SMA32_Slope,RSI,IHR,Dark_Cloud,Shooting_Star,Bear_Engulf,Iday_Range,ADR,Close_%DHigh,SMA16,SMA32,S_R,YRA_Diff,YRA_STD,R_Tested,R_Broken,S_Tested,S_Broken,H1_Range,To_S_0,To_S_1,To_S_2
139,2026-02-06 04:45:00-05:00,1.357710,-0.004655,0.000525,-0.003185,-0.000530,-0.000535,0.000525,0.000000,0.000000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.000565,0.000897,0.938053,2,26.755719,40.640592,62.571942,NaN,NaN,NaN,NaN,0.007380,0.008918,0.071816,1.357068,1.356202,NaN,0.005152,0.003523,0,1,0,0,0.001650,NaN,NaN,NaN
140,2026-02-06 10:00:00-05:00,1.360605,-0.001760,0.001225,-0.000290,-0.000815,-0.000840,0.001225,0.001183,0.000000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.000840,0.000853,0.970238,2,49.423970,39.593584,67.920009,NaN,NaN,NaN,NaN,0.010560,0.008918,0.077178,1.359280,1.358267,NaN,0.005152,0.003523,0,1,0,0,0.002410,NaN,NaN,NaN
141,2026-02-06 11:00:00-05:00,1.360905,-0.001460,0.000740,0.000010,-0.000655,-0.000815,0.000740,0.000721,0.000000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.001155,0.001065,0.567100,2,46.453895,43.136503,63.125470,NaN,NaN,NaN,NaN,0.010700,0.008918,0.061215,1.359738,1.358728,NaN,0.005152,0.003523,0,1,0,0,0.002180,NaN,NaN,NaN
142,2026-02-09 06:15:00-05:00,1.361500,-0.008550,0.000055,-0.007780,-0.000805,-0.001890,0.000055,0.000000,0.000000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.001285,0.001132,0.626459,1,42.542862,14.819918,55.309325,NaN,True,NaN,NaN,0.004250,0.008957,0.328235,1.360470,1.360627,NaN,0.002548,0.003523,1,1,1,1,0.002980,NaN,NaN,NaN
143,2026-02-11 04:00:00-05:00,1.368605,-0.002650,0.007645,0.005855,-0.000780,-0.001200,0.000195,0.000000,0.000000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.000980,0.000800,0.795918,2,48.550195,44.075221,67.204611,NaN,NaN,NaN,True,0.006235,0.009128,0.125100,1.367482,1.366593,NaN,-0.003073,0.003543,1,1,0,0,0.002470,NaN,NaN,NaN
144,2026-02-18 02:00:00-05:00,1.355725,-0.002490,0.006560,0.006250,-0.002150,-0.002335,0.000345,0.000096,-0.000081,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.002190,0.000520,0.981735,1,12.509848,-8.180230,49.807330,True,True,True,NaN,0.002890,0.009051,0.743945,1.355637,1.355806,1.0,0.004884,0.003716,4,2,0,2,0.002420,NaN,NaN,NaN
145,2026-02-25 21:00:00-05:00,1.357065,-0.000120,0.012525,0.008850,0.000000,0.000590,0.012525,0.000869,0.001290,0.00214,0.00200,0.00496,0.00149,0.00149,0.00149,0.000560,0.000514,0.794643,2,28.719069,15.209658,67.148904,NaN,NaN,NaN,True,0.002445,0.009293,0.182004,1.356128,1.355746,NaN,-0.001788,0.003723,0,0,0,0,0.001045,0.00211,0.00516,NaN
146,2026-03-06 08:45:00-05:00,1.333850,-0.007780,0.001320,-0.006230,-0.003810,-0.004130,0.001320,0.000277,-0.000815,0.00000,-0.00075,0.00000,0.00000,0.00000,0.00000,0.004115,0.001975,0.925881,1,52.073538,-4.111065,48.065741,NaN,NaN,NaN,NaN,0.007940,0.010079,0.660579,1.333573,1.334665,4.0,-0.001119,0.003966,2,2,0,2,0.006520,-0.00075,NaN,NaN
147,2026-03-09 15:45:00-04:00,1.342940,-0.001775,0.000530,-0.000450,-0.001630,-0.000935,0.000530,0.000000,0.000000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.001690,0.001246,0.964497,2,60.689993,61.532578,68.540948,NaN,NaN,NaN,NaN,0.016260,0.009884,0.100246,1.339856,1.338511,NaN,0.000591,0.003713,2,4,2,2,0.006205,NaN,NaN,NaN
148,2026-03-10 02:45:00-04:00,1.345030,-0.003310,0.003805,0.003225,-0.000745,-0.002365,0.000320,0.000000,0.000000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.001135,0.000777,0.656388,1,49.774958,26.133723,63.597555,NaN,True,NaN,NaN,0.004415,0.010196,0.168743,1.343000,1.343020,3.0,0.006209,0.003829,0,2,2,1,0.002185,NaN,NaN,NaN


## Signal Stats

### Initial Signal Stats

In [653]:
# get the total number of bear_bbr signals
total_signals_count = bearish_bbr_df[bearish_bbr_df.columns[0]].count()

day_closed_lower = bearish_bbr_df.query("D_Gain > 0")
day_closed_higher = bearish_bbr_df.query("D_Gain <= 0")
day_closed_lower_count = day_closed_lower["D_Gain"].count()
day_closed_higher_count = day_closed_higher["D_Gain"].count()
day_closed_lower_pct = round(day_closed_lower_count / total_signals_count * 100,2)
day_closed_lower_pips = bearish_bbr_df["D_Gain"].sum().round(6)

# filter the dataframe for occurences where after the signal
# the price moved lower more than it moved higher
d_max_down_greater = bearish_bbr_df.query("D_Max_Up.abs() < D_Max_Down")
# get the count of occurences where price dropped more then it rose after signal
d_max_down_greater_count = d_max_down_greater[d_max_down_greater.columns[0]].count()

# percentage of where price fell further than the max it went against the entry price
d_max_down_greater_pct = round(d_max_down_greater_count / total_signals_count * 100, 2)

max_down_win = bearish_bbr_df.query("Max_Down > Max_Up.abs()")["Date"].count()
max_down_loss = bearish_bbr_df.query("Max_Down <= Max_Up.abs()")["Date"].count()
total_max_down_trades = max_down_win + max_down_loss
max_down_win_rate = round((max_down_win / total_max_down_trades) * 100 ,2)

max_down_pips = bearish_bbr_df["Max_Down"].sum().round(6)
max_up_pips = bearish_bbr_df["Max_Up"].sum().round(6)
total_max_down_gain = max_down_pips + max_up_pips

average_max_up = bearish_bbr_df["Max_Up"].mean().round(6)
average_max_down = bearish_bbr_df["Max_Down"].mean().round(6)


print(
    f"""
    ========= Initial Bear_BBR Statistics =============

    Total Bear_BBR Signals: {total_signals_count}

    ** Where the move down from the signal resulted **
    ** in a positive gain at the close of the day **
    Day Closed Lower (Count): {day_closed_lower_count}
    Day Closed Higher (Count): {day_closed_higher_count}
    Day Closed Lower (%): {day_closed_lower_pct}
    Day Closed Lower (Pips): {day_closed_lower_pips}

    ** Where after the signal the max move down on the day **
    ** was greater than the max move up **
    D_Max_Down > D_Max_Up (Count): {d_max_down_greater_count}
    D_Max_Down > D_Max_Up (%): {d_max_down_greater_pct}

    ** Where the max move down after the signal **
    ** was greater than the max move up (exit if price closes above signal high) **
    Max_Down > Max_Up (Count): {max_down_win}
    Max_Down <= Max_Up (Count): {max_down_loss}
    Total Trades (Max_Down): {total_max_down_trades}
    Max_Down > Max_Up (Win Rate): {max_down_win_rate}

    Max_Down Pips: {max_down_pips}
    Max_Up Pips: {max_up_pips}
    Total Max_Down Gain: {total_max_down_gain}
    
    Average Max_Up Pips: {average_max_up}
    Average Max_Down Pips: {average_max_down}
 
    ==============================================================================
    """
)


    ========= Initial Bear_BBR Statistics =============

    Total Bear_BBR Signals: 149

    ** Where the move down from the signal resulted **
    ** in a positive gain at the close of the day **
    Day Closed Lower (Count): 72
    Day Closed Higher (Count): 77
    Day Closed Lower (%): 48.32
    Day Closed Lower (Pips): -0.06571

    ** Where after the signal the max move down on the day **
    ** was greater than the max move up **
    D_Max_Down > D_Max_Up (Count): 71
    D_Max_Down > D_Max_Up (%): 47.65

    ** Where the max move down after the signal **
    ** was greater than the max move up (exit if price closes above signal high) **
    Max_Down > Max_Up (Count): 66
    Max_Down <= Max_Up (Count): 82
    Total Trades (Max_Down): 148
    Max_Down > Max_Up (Win Rate): 44.59

    Max_Down Pips: 0.29249
    Max_Up Pips: -0.156265
    Total Max_Down Gain: 0.13622500000000004
    
    Average Max_Up Pips: -0.001056
    Average Max_Down Pips: 0.001976
 
    


### Summary

- From the initial numbers, we can see that the Bear_BBR signal occurred `359` times.
- After the signal occuring, the *max price move down* was greater than the *max price move up* on the day, `47.63%` of the time.
- When the signal occurred, `46.63%` of the time the price dropped more than it gained after the signal, before closing above the signal high.
- The total gains where the price fell more than it rose after the signal (before closing above the signal high) was `3611` pips.
- The average number of pips where price moved against the signal was `10.32` pips
- The average number of pips gained before price closed above the signal high was `20.46` pips.

#### What does this tell us?

- We can see just under 50% of the time, the price moves lower after the signal, *before or without closing above the signal high*.
- Additionally, the gains from when the price moves in favour of the signal is greater than when the price moves against the signal. In short the signal generates more gains than losses overall.
- On average, the price moves down nearly double the amount it moves against the signal.

#### What information is missing?

- Although the signal gains more pips overall, this doesn't tell us the best place to take profit to capture the max move down.
- It is not clear what factors make a move down more likely, or when the signal is more likely to fail.


## Investigating factors that increase signal accuracy

### Holding a short position from signal entry to close of the day



In [654]:
def daily_pip_gain(df: Series, pct_adr: int):
    d_pip_gain = df["D_Gain"] if abs(df["D_Max_Up"]) < (df["ADR"] * pct_adr) else (df["ADR"] * -pct_adr)
    return d_pip_gain

def condition_stats(df: DataFrame, pct_adr: int):

    df["Pip_Gain"] = df.apply(daily_pip_gain, axis=1, args=[pct_adr])
    df["ADR%"] = df.apply(lambda x: x["ADR"] * pct_adr, axis=1)
    
    success_series = df.query(f"D_Max_Up.abs() < (ADR * {pct_adr})")["Pip_Gain"]
    fail_series = df.query(f"D_Max_Up.abs() >= (ADR * {pct_adr})")["Pip_Gain"]

    success_total = success_series.count()
    fail_total = fail_series.count()
    total_trades = success_total + fail_total
    
    win_rate = round(success_total / total_trades * 100,2)
    condition_total_pips = round(success_series.sum() + fail_series.sum(),6)

    return {
        "Pct_ADR": pct_adr,
        "Pct_ADR_Avg": df["ADR%"].mean(),
        "ADR_Avg": df["ADR"].mean(),
        "Total Trades": total_trades,
        "Win": success_total,
        "Loss": fail_total,
        "Win Rate": win_rate,
        "Avg_win": success_series.mean(),
        "Avg_Loss": fail_series.mean(),
        "Total Pips": condition_total_pips
    }
    
# columns
cols1 = [
    "Date", "Entry_Price", "Pip_Gain", "D_Max_Up", "ADR%", "D_Gain", "ADR"]
#df
d_gain_df = bearish_bbr_df


#### Using a stop loss set to a percentage of the ADR

Since we already know from the intial stats that the price will close lower on the day 48% of the time after the signal, we should try to limit the losses for the 52% of the times where it fails.

A `stop loss` is used to limit losses, and can be static (e.g. 10 pips) or dynamic (e.g. set as multiple of the `ATR`, or a trailing stop ... etc.).
The ATR takes into account the volatility of the last *x* candles, so is usually a good choice. 

For this particular scenario however, when the `Bear_BBR` signal occurs we are looking to hold a short position until the close of the day. The ATR is not the best approach here, because ATR will look at the average candle range within increments of the timeframe period (e.g. 15min) over *x* period of candles. 

To address this I will use a percentage of the the average daily range (**ADR**) as a stop loss. The ADR calculates the average daily price movement over last 30 days of trading, which will tell me the average number of pips the price moves within a day over that period.

To find a reasonable stop loss range, I'll iterate through different ADR percentages from 1 to 100.

In [655]:
d_gain_experimental_sl = [condition_stats(d_gain_df, x * 0.01) for x in range(1,101)]
d_gain_ex_sl_res_df = pd.DataFrame(d_gain_experimental_sl)
d_gain_ex_sl_res_df

,Pct_ADR,Pct_ADR_Avg,ADR_Avg,Total Trades,Win,Loss,Win Rate,Avg_win,Avg_Loss,Total Pips
0,0.01,0.000089,0.008926,128,6,122,4.69,0.002009,-0.000090,0.001112
1,0.02,0.000179,0.008926,128,9,119,7.03,0.003189,-0.000179,0.007345
2,0.03,0.000268,0.008926,128,11,117,8.59,0.003883,-0.000269,0.011257
3,0.04,0.000357,0.008926,128,13,115,10.16,0.003599,-0.000359,0.005522
4,0.05,0.000446,0.008926,128,16,112,12.50,0.003651,-0.000449,0.008105
5,0.06,0.000536,0.008926,128,18,110,14.06,0.003440,-0.000540,0.002542
6,0.07,0.000625,0.008926,128,19,109,14.84,0.003460,-0.000629,-0.002878
7,0.08,0.000714,0.008926,128,20,108,15.62,0.003648,-0.000719,-0.004652
8,0.09,0.000803,0.008926,128,22,106,17.19,0.003617,-0.000806,-0.005865
9,0.10,0.000893,0.008926,128,24,104,18.75,0.003461,-0.000895,-0.010039


- This shows the stop_loss is directly related to the win rate.
- It also shows that even with a higher win rate, it does not mean a higher `total pips` value. This makes sense because the larger the stop loss, the larger each loss will be.
- The ideal stop loss will be as low as possible and yield a reasonably high `total pips`.
- The maximum total pips returned was 0.011257 which was a result of 3% ADR stop loss, however we can't take this at face value because it could be an outlier. We also need to consider that when a trade is placed, a percentage of equity will be risked (e.g. 1%), so a lower ADR percentage will allow for a larger trade size to be used, which maximises the monetary return.
- We should look at the top ten `total pips` values and see what range of ADR percentages provided the best return.

In [656]:
top_ten_returns_by_adr_pct = d_gain_ex_sl_res_df["Total Pips"].nlargest(10)
top_ten_returns_by_adr_pct

2     0.011257
15    0.009850
32    0.009413
19    0.008731
16    0.008517
4     0.008105
17    0.007830
1     0.007345
21    0.006803
47    0.006492
Name: Total Pips, dtype: float64

The output shows the top ten `total pips` values by their index. Since the percent of ADR value is the `index+1`, we can see what contiguous index values provide the best return. I prefer contiguous values to get a better approximation of ADR percent ranges that work well, which should help to avoid outliers that may have been a fluke. 

From the output there's 3 contiguous index values between 15-17 which map to 16-18% ADR.

When you look at 16-18% ADR, the win rate is (33.59, 37.50, 39.06) but the average win for 16% ADR (0.003070) is double the average loss (0.001437). The 16% ADR value also yields the second highest `total pips` return out of all ADR percentage values. So 16% ADR looks good becauses it keeps the pip loss small, and still has a reasonable win rate which we can try to improve.

The reason for choosing the 2nd lowest ADR percentage value is that is allows us to take a larger trade size while risking the same equity percentage. A win with a larger trade value will generate a larger monetary return for the total pips. The reason for discounting 3% ADR is because of the terrible win rate, and the fact it is likely to be specifically good for this current dataset. 

To check the potential of monetary rerturns using this strategy, I'll simulate an account size of $100,000 and a trade risk value of 1%. 

In [657]:
risk_ex_df = d_gain_ex_sl_res_df
risk_ex_df["Account_Size"] = risk_ex_df.apply(lambda x: 100000, axis=1)
risk_ex_df["Risk"] = risk_ex_df.apply(lambda x: 0.01, axis=1)
risk_ex_df["Trade_Value"] = risk_ex_df.apply(lambda x: round((x["Account_Size"] * x["Risk"])/x["Pct_ADR_Avg"],2), axis=1)
risk_ex_df["Return"] = risk_ex_df.apply(lambda x: round(x["Trade_Value"] * x["Total Pips"],2), axis=1)

risk_ex_df.filter(risk_ex_df["Return"].nlargest(10).index, axis=0)

,Pct_ADR,Pct_ADR_Avg,ADR_Avg,Total Trades,Win,Loss,Win Rate,Avg_win,Avg_Loss,Total Pips,Account_Size,Risk,Trade_Value,Return
2,0.03,0.000268,0.008926,128,11,117,8.59,0.003883,-0.000269,0.011257,100000,0.01,3734606.69,42040.47
1,0.02,0.000179,0.008926,128,9,119,7.03,0.003189,-0.000179,0.007345,100000,0.01,5601910.03,41146.03
4,0.05,0.000446,0.008926,128,16,112,12.50,0.003651,-0.000449,0.008105,100000,0.01,2240764.01,18161.39
3,0.04,0.000357,0.008926,128,13,115,10.16,0.003599,-0.000359,0.005522,100000,0.01,2800955.02,15466.87
0,0.01,0.000089,0.008926,128,6,122,4.69,0.002009,-0.000090,0.001112,100000,0.01,11203820.07,12458.65
15,0.16,0.001428,0.008926,128,43,85,33.59,0.003070,-0.001437,0.009850,100000,0.01,700238.75,6897.35
16,0.17,0.001517,0.008926,128,48,80,37.50,0.002733,-0.001534,0.008517,100000,0.01,659048.24,5613.11
19,0.20,0.001785,0.008926,128,54,74,42.19,0.002639,-0.001808,0.008731,100000,0.01,560191.00,4891.03
17,0.18,0.001607,0.008926,128,50,78,39.06,0.002688,-0.001623,0.007830,100000,0.01,622434.45,4873.66
5,0.06,0.000536,0.008926,128,18,110,14.06,0.003440,-0.000540,0.002542,100000,0.01,1867303.34,4746.69


The results show an ADR percentage of 1% yields the highest return but has a 3% win rate. This is clearly an anomaly because 2 to 3% ADR would also yield high returns if this was a good area for a stop loss. 

The better indicator is the range between 16-18%, which is a countiguous range of values with the highest monetary return out of the distribution. Based on this I'll view 16% ADR as an ideal stop loss and look for ways to improve the win rate.

#### Testing if the slope angle increases win rate

Considering the Bear_BBR is a short signal, we want to avoid cases were there is a strong uptrend. There's probably many ways to define a strong uptrend, but I think looking at the angle of the simple moving average (SMA) slopes is a way to define it. Generally, a strong trend or up-move will have SMA's pointing up. To quantify "pointing up" we'll say if the angle is greater than *x* the trend is too bullish, so skip the signal.

I think setting the angle limit to 45 degrees makes sense. Let's see what this shows:

In [659]:
# conditions
slope_lt_45_df = bearish_bbr_df.query("SMA32_Slope < 45").copy()
slope_lt_45_res = condition_stats(slope_lt_45_df, 0.16)
pd.DataFrame([slope_lt_45_res])

,Pct_ADR,Pct_ADR_Avg,ADR_Avg,Total Trades,Win,Loss,Win Rate,Avg_win,Avg_Loss,Total Pips
0,0.16,0.001418,0.00886,98,33,65,33.67,0.003331,-0.001421,0.017528


This shows a slight reduction in the total number of trades (98), which we'd expect with this filter. We also see a similar win rate, and an increase of pips gained, making the `total pips` 175. It definitely looks better, but we can also test for a range of angles betweem 90 and 0, to see of there is anything better.

In [660]:
slope_angle_ex = [condition_stats(bearish_bbr_df.query(f"SMA32_Slope < {x}").copy(), 0.16) for x in range(0,91)]
slope_angle_ex_df = pd.DataFrame(slope_angle_ex)
slope_angle_ex_df

,Pct_ADR,Pct_ADR_Avg,ADR_Avg,Total Trades,Win,Loss,Win Rate,Avg_win,Avg_Loss,Total Pips
0,0.16,0.001480,0.009248,13,4,9,30.77,0.003280,-0.001524,-0.000595
1,0.16,0.001480,0.009248,13,4,9,30.77,0.003280,-0.001524,-0.000595
2,0.16,0.001463,0.009143,14,5,9,35.71,0.003593,-0.001524,0.004250
3,0.16,0.001463,0.009143,14,5,9,35.71,0.003593,-0.001524,0.004250
4,0.16,0.001453,0.009082,15,6,9,40.00,0.003295,-0.001524,0.006055
5,0.16,0.001461,0.009134,16,7,9,43.75,0.003359,-0.001524,0.009795
6,0.16,0.001465,0.009158,17,7,10,41.18,0.003359,-0.001524,0.008269
7,0.16,0.001465,0.009158,17,7,10,41.18,0.003359,-0.001524,0.008269
8,0.16,0.001465,0.009158,17,7,10,41.18,0.003359,-0.001524,0.008269
9,0.16,0.001446,0.009040,19,8,11,42.11,0.003050,-0.001496,0.007940


We can view the angles with the largest returns:

In [662]:
slope_angle_ex_df["Account_Size"] = slope_angle_ex_df.apply(lambda x: 100000, axis=1)
slope_angle_ex_df["Risk"] = slope_angle_ex_df.apply(lambda x: 0.01, axis=1)
slope_angle_ex_df["Trade_Value"] = slope_angle_ex_df.apply(lambda x: round((x["Account_Size"] * x["Risk"])/x["Pct_ADR_Avg"],2), axis=1)
slope_angle_ex_df["Return"] = slope_angle_ex_df.apply(lambda x: round(x["Trade_Value"] * x["Total Pips"],2), axis=1)
slope_angle_ex_df.filter(slope_angle_ex_df["Total Pips"].nlargest(10).index, axis=0)

,Pct_ADR,Pct_ADR_Avg,ADR_Avg,Total Trades,Win,Loss,Win Rate,Avg_win,Avg_Loss,Total Pips,Account_Size,Risk,Trade_Value,Return
29,0.16,0.001416,0.008852,52,23,29,44.23,0.003205,-0.001439,0.031976,100000,0.01,706024.45,22575.84
30,0.16,0.001414,0.008840,54,23,31,42.59,0.003205,-0.001434,0.029254,100000,0.01,707049.12,20684.01
28,0.16,0.001413,0.008832,48,21,27,43.75,0.003219,-0.001439,0.028733,100000,0.01,707658.99,20333.17
40,0.16,0.001415,0.008845,89,32,57,35.96,0.003385,-0.001416,0.027597,100000,0.01,706612.11,19500.37
31,0.16,0.001415,0.008845,57,24,33,42.11,0.003122,-0.001440,0.027411,100000,0.01,706612.50,19368.96
25,0.16,0.001398,0.008738,45,20,25,44.44,0.003126,-0.001409,0.027279,100000,0.01,715297.58,19512.60
26,0.16,0.001398,0.008738,45,20,25,44.44,0.003126,-0.001409,0.027279,100000,0.01,715297.58,19512.60
32,0.16,0.001404,0.008778,61,25,36,40.98,0.003131,-0.001426,0.026960,100000,0.01,712031.23,19196.36
41,0.16,0.001415,0.008846,90,32,58,35.56,0.003385,-0.001416,0.026170,100000,0.01,706547.34,18490.34
33,0.16,0.001408,0.008798,65,26,39,40.00,0.003139,-0.001427,0.025948,100000,0.01,710350.22,18432.17


This shows angles between the range of 28 and 30 have returns above 287 pips, however the best fit is **33** degrees. From this sample, if we only took `Bear_BBR` trades with an SMA32 slope under 29 degrees the `total pips` would be 319.7 with a monetary return of $22575.84 (excluding fees and commissions). 

The results look good, but this only shows filtering for trades which are held until the close of the day. It relies on the day being a complete reversal day and also using a tighter stop loss (16% ADR). These two factors may be unique to this dataset so we need to test over a different year and see if the results are similar.

When I run the tests over the 2025 period with file `FE_V2_GBPUSD_15mins_1yr_End_20250311`, it turns out the optimum stop loss is 2% ADR which gives a 3.64% win rate, but does deliver the highest return (~12k). Obviously this is unrealistic as we have to account for the spread + commissions, so the actual return is probably far less. Outside of that, if we look at the top ten highest `total pips`, the next best stop loss is 29% ADR, which is more reasonable and carries a 50% win rate and 280 pip gain over 147 trades (when sma32_slope < 50).

So comparing the two we can consider:
- The approach to calculating the stop loss is biased to the dataset, which is why the best stop loss (by gains) differs between 2024-2025 and 2025-2026. When looking at common ADR% stop loss values across both data sets, 25-30 seems to appear in both, so would be a good realistic start.
- The optimal slop value also changes, with the optimal slop being under 5 degress in 2024-2025 dataset. The slop is considered in the signal definition as well, so I think this reading is probably biased to the dataset.
- The max move down after the signal was greater than the max move up 47% of the time on both datasets. This shows the signals are good and are correctly identifying a drop in price at least half the time.

Since bollinger bands can be used for breakouts and mean reversion, it does explain why there's ~50% accuracy. In FX there are a lot of days where price moves in a range, or has low trend movement. This means even though a signal predicts a successful fall in price, the price could still reverse and close above the signal candle. To solve this we need to define a take profit target that will maximise the gains on a successful signal. When I briefly investigated strategies for targets by manual chart analysis, there appears to be a relationship between the high of the signal candle and the intraday range. I will delve into this further in the next section.